# MinecraftWorldModel 训练 Demo（Δz-JEPA · 序列对齐后果世界模型）

从 OpenAI BASALT 离线演示数据（`.mp4` + `.jsonl`）训练 `MinecraftWorldModel`：
采用后果加权潜对齐、多上下文一致性及 Sliced 正则（SIGReg）防表征坍缩的后果结构世界模型。

**判断训练有效的北极星指标**：
- **`align` 越低越好**：代表世界模型潜空间预测误差跑赢了 persistence（零运动）基线。
- **`agree` 越低越好**：代表多上下文截止对相同未来时刻预测的一致性提升，脑内推演更加稳定。
- **`corr(w,future)` 越大越好（正相关）**：证明后果权重 $w$ 成功追踪了未来的潜状态发散幅度。
- **`corr(w,pixel)` 越小越好（接近 0）**：表明后果权重与局部像素差不相关，说明模型真正关注“后果”，而非“像素面积”，没有发生假成功。
- **`recon_inv` 指标下降**：说明不可逆通道信息重构精度增加。

**安全提示**：密钥一律从 **Colab Secrets**（左侧 🔑）注入（见 §1），不再硬编码——本文件已脱敏、可入库。
clone 用的 PAT 若曾明文出现过，请去 github.com/settings/tokens 撤销/轮换；wandb key 同理（wandb.ai/authorize）。

In [ ]:
# ===== §1 配置 =====
import json, os, requests

BASE = "https://openaipublic.blob.core.windows.net/minecraft-rl"
INDEX = {
    "FindCave":      f"{BASE}/snapshots/find-cave-Jul-28.json",
    "MakeWaterfall": f"{BASE}/snapshots/waterfall-Jul-28.json",
    "AnimalPen":     f"{BASE}/snapshots/pen-animals-Jul-28.json",
    "BuildHouse":    f"{BASE}/snapshots/build-house-Jul-28.json",
}
TASKS = ["FindCave", "MakeWaterfall", "AnimalPen", "BuildHouse"]   # 四任务混采

DISK_KEEP  = 64          # 训练池磁盘滑动窗口(段);64 段 ≈ 5GB。滚动下载器在四个
                         # 任务的全量索引上循环,旧的删掉防占满
DL_THREADS = 8           # 后台滚动下载线程数
SIZE       = 128         # 训练分辨率(VPTStreamDataset 解码时 on-the-fly resize)
CAM_SCALE  = 20.0        # 相机归一化尺度(度/帧;§2 已把原始像素×360/2400 转成度)

# 密钥从 Colab Secrets(左侧 🔑 图标)注入
try:
    from google.colab import userdata
    PAT       = userdata.get("GH_PAT")
    WANDB_KEY = userdata.get("WANDB_API_KEY")
except Exception:   
    from dotenv import load_dotenv                         # 非 Colab 环境回退到环境变量
    load_dotenv()
    PAT       = os.environ.get("GH_PAT", "")
    WANDB_KEY = os.environ.get("WANDB_API_KEY", "")
WANDB_PROJECT = "minecraft-world-model"     # wandb 项目名
WANDB_RUN    = None                         # run 名,None = 自动生成

REPO     = "OopsYouDiedE/tao-not-42-base-refactor-world-model-contract"
REPO_DIR = "/content/repo"
OUT      = "/content/runs/data"              # 统一数据根目录(holdout 段文件名以 z_hold_ 前缀排在末尾)

ALL = {}                                    # task -> (basedir, relpaths)
for _t in TASKS:
    _idx = requests.get(INDEX[_t], timeout=60).json()
    ALL[_t] = (_idx["basedir"].rstrip("/"), _idx["relpaths"])
    print(f"  {_t}: {len(ALL[_t][1])} 段")
print(f"四任务混采  滑动窗口={DISK_KEEP} 段  holdout=每任务 1 段  CAM_SCALE={CAM_SCALE}")

In [ ]:
# ===== §2 数据准备: 统一数据池 + z_hold_ 前缀划分 holdout 集合 =====
# 架构:后台 DL_THREADS 个线程在全量索引上随机循环下载并转成 jsonl → OUT。
# 4 段 holdout clip 带有 z_hold_ 文件名前缀，使文件名排序时自动排列到末尾。
# 在 train_minecraft 启动时指定 --holdout_n 4，数据集将自动截取末尾 4 段作为 holdout 集，其余为训练集。
import threading, random
from concurrent.futures import ThreadPoolExecutor

os.makedirs(OUT, exist_ok=True)

# 旧布局清理
for f in os.listdir(OUT):
    p = os.path.join(OUT, f)
    if os.path.isfile(p) and ".part" in f:
        os.remove(p)

PX2DEG = 360.0 / 2400.0          # VPT CAMERA_SCALER:鼠标像素 → 视角度数

_KEYMAP = {
    "key.keyboard.w": "key_w", "key.keyboard.s": "key_s",
    "key.keyboard.a": "key_a", "key.keyboard.d": "key_d",
    "key.keyboard.space": "key_space",
    "key.keyboard.left.shift": "key_sneak",
    "key.keyboard.left.control": "key_sprint",
    "key.keyboard.q": "key_drop", "key.keyboard.e": "key_inventory",
    **{f"key.keyboard.{i}": f"key_hotbar.{i}" for i in range(1, 10)},
}
_BTN = {0: "key_attack", 1: "key_use"}
TASK_TEXT = {"FindCave": "find a cave", "MakeWaterfall": "make a waterfall",
             "AnimalPen": "build an animal pen", "BuildHouse": "build a village house"}

def _size(p):
    try:
        return os.path.getsize(p)
    except OSError:
        return -1

def _mtime(p):
    try:
        return os.path.getmtime(p)
    except OSError:
        return 0.0

def prepare(task, rel, out_dir, is_holdout=False):
    basedir = ALL[task][0]
    raw_name = os.path.basename(rel)[:-4]
    name = f"z_hold_{raw_name}" if is_holdout else raw_name
    mp4, jsonl = f"{out_dir}/{name}.mp4", f"{out_dir}/{name}.jsonl"
    tmp = f".part{threading.get_ident()}"

    if _size(mp4) <= 0:                                             # mp4
        r = requests.get(f"{basedir}/{rel}", stream=True, timeout=300); r.raise_for_status()
        with open(mp4 + tmp, "wb") as f:
            for chunk in r.iter_content(1 << 22):
                f.write(chunk)
        os.replace(mp4 + tmp, mp4)

    if not os.path.exists(jsonl):                                   # jsonl
        r = requests.get(f"{basedir}/{rel[:-4]}.jsonl", timeout=300); r.raise_for_status()
        with open(jsonl + tmp, "w", encoding="utf-8") as fw:
            for i, line in enumerate(r.text.splitlines()):
                d = json.loads(line)
                gui = bool(d.get("isGuiOpen", False))
                kb = {_KEYMAP[k]: 1 for k in d.get("keyboard", {}).get("keys", []) if k in _KEYMAP}
                m = d.get("mouse", {})
                if not gui:
                    kb.update({_BTN[b]: 1 for b in m.get("buttons", []) if b in _BTN})
                dx = 0.0 if gui else m.get("dx", 0.0) * PX2DEG
                dy = 0.0 if gui else m.get("dy", 0.0) * PX2DEG
                frame = {"keyboard": kb, "mouse": {"dx": dx, "dy": dy}}
                if gui:
                    frame["gui"] = 1
                if i == 0:
                    frame["task"] = TASK_TEXT[task]
                fw.write(json.dumps(frame, ensure_ascii=False) + "\n")
        os.replace(jsonl + tmp, jsonl)
    return name

# --- holdout: 每任务 1 段(前缀为 z_hold_ 确保排在末尾) ---
HOLDOUT = [(t, ALL[t][1][-1]) for t in TASKS]
with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(lambda tr: prepare(tr[0], tr[1], OUT, is_holdout=True), HOLDOUT))
print(f"holdout ready: {len(HOLDOUT)} clips(已加 z_hold_ 前缀) in {OUT}")

# --- train: 后台滚动下载 (自动排除 holdout 段) ---
_HOLD_SET = {rel for _, rel in HOLDOUT}
POOL = [(t, rel) for t in TASKS for rel in ALL[t][1] if rel not in _HOLD_SET]

def _evict():
    # 只对非 holdout (不含 z_hold_ 前缀) 的文件执行淘汰，防止误删 holdout 数据
    mp4s = sorted((os.path.join(OUT, f) for f in os.listdir(OUT)
                   if f.endswith(".mp4") and not f.startswith("z_hold_")), key=_mtime)
    for old in mp4s[:max(0, len(mp4s) - DISK_KEEP)]:
        for p in (old, old[:-4] + ".jsonl"):
            try:
                os.remove(p)
            except OSError:
                pass

def _roller(tid):
    rng = random.Random(1234 + tid)
    while True:
        try:
            t, rel = POOL[rng.randrange(len(POOL))]
            prepare(t, rel, OUT, is_holdout=False)
            _evict()
        except Exception as e:
            print(f"[dl-{tid}] {type(e).__name__}: {e}")

if not globals().get("_DL_STARTED"): 
    _DL_STARTED = True
    for i in range(DL_THREADS):
        threading.Thread(target=_roller, args=(i,), daemon=True).start()
n_now = sum(f.endswith('.mp4') and not f.startswith('z_hold_') for f in os.listdir(OUT))
print(f"rolling downloader active: {DL_THREADS} 线程 → {OUT} (磁盘保留 {DISK_KEEP} 段，当前 {n_now} 段)。")

In [ ]:
# ===== §3 训练 · 后果结构自监督世界模型 =====
# 骨干用开放权重 dinov2 (--config configs/minecraft/dinov2.yaml, 无需 HF token)。
# 使用新版 train_minecraft.py，指定单一 --data_dir {OUT} 并设置 --holdout_n 4，
# 模型会自动用排序后的末尾 4 个 z_hold_ 文件作为 holdout，其余作为训练。
import os

if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    !rm -rf {REPO_DIR}
    !git clone --depth=1 https://{PAT}@github.com/{REPO}.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch --depth=1 origin main && git reset --hard origin/main
!cd {REPO_DIR} && git log --oneline -1

# 同步依赖
!cd {REPO_DIR} && grep -ivE '^(torch|torchvision|xformers)' requirements.txt > /tmp/req_colab.txt && pip install -q -r /tmp/req_colab.txt
os.environ["WANDB_API_KEY"] = WANDB_KEY

# 启动训练 (传入已重新支持的 W&B 实验同步参数)
# 显存: 训练步峰值 ∝ B·T (SDPA 省显存注意力, 无 O(L²) 爆炸)。本机实测标度律
#   reserved(GB) ≈ 0.26 + 32.8·(B·T/1000)。L4(24GB) 留 ~4GB 余量 → B·T≈512 (~18GB)。
#   注意: 旧的 batch128×seq60 (B·T=7680) 峰值 ~200GB, 任何单卡都放不下。
# --viz_every 20: 每 20 epoch 渲染一张 Δz-JEPA 诊断面板, 与 eval 同步并上传 W&B。
!cd {REPO_DIR} && python train/minecraft/train_minecraft.py \
  --config configs/minecraft/dinov2.yaml \
  --data_dir {OUT} \
  --img_size 128 --batch 16 --seq_len 32 --frame_skip 8 \
  --steps_per_epoch 5 --epochs 300 --lr 3e-4 \
  --beta_sigreg 0.03 \
  --log_every 1 --eval_every 20 --viz_every 20 --viz_dir runs/mc_viz \
  --clip_cache 4 --clip_refresh 256 --holdout_n 4 \
  --ckpt_dir runs/mc_ckpt \
  --device cuda \
  --wandb \
  --wandb_project {WANDB_PROJECT} \
  --wandb_run seqalign-consequence-jepa-128px

In [ ]:
# ===== §4 仅评估 (不训练): 交互式加载最佳 ckpt 进行精度复核 =====
# 本格使用 Jupyter 交互方式直接运行评估，避免了命令行脚本不支持 --eval_only 参数的问题。
# 前置: §1 配置 + §2 完毕。只需几秒钟即可输出评估结果。
import os, sys, glob, torch
from torch.utils.data import DataLoader

ARTIFACT = "best-model-seqalign-consequence-jepa-128px:latest"   # ← 想看哪个 run 的 best 改这里
os.environ["WANDB_API_KEY"] = WANDB_KEY

# 确保本地代码路径正确
if not os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git clone --depth=1 https://{PAT}@github.com/{REPO}.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git fetch --depth=1 origin main && git reset --hard origin/main

import wandb
art = wandb.Api().artifact(f"{WANDB_PROJECT}/{ARTIFACT}", type="model")
ckpt_path = glob.glob(f"{art.download(root=f'{REPO_DIR}/runs/mc_ckpt')}/**/*.pt", recursive=True)[0]
print("Loading ckpt:", ckpt_path)

# 导入模块
sys.path.insert(0, REPO_DIR)
from net.config import ModelConfig
from net.world_model import MinecraftWorldModel
from net.effect_tokenizer import EffectTokenizer
from domains.minecraft.vpt_dataset import VPTStreamDataset
from train.minecraft.eval import evaluate

state = torch.load(ckpt_path, map_location="cuda")
cfg_dict = state["config"]
cfg = ModelConfig.from_dict(cfg_dict)
cfg.max_skip = 8

model = MinecraftWorldModel(cfg).to("cuda")
model.load_state_dict(state["model"], strict=False)
model.eval()

effect_tok = EffectTokenizer(d_inv=cfg.d_inv, event_vocab_size=cfg.effect.event_vocab_size).to("cuda")
effect_tok.load_state_dict(state["effect_tok"])
effect_tok.eval()

# 载入 holdout 数据集 (4 段)。seq_len 与训练 (§3) 对齐为 32，使 align/agree 口径一致；
# 仅评估为 no_grad，batch 16 不会 OOM。
eval_ds = VPTStreamDataset(OUT, seq_len=32, fps=20, img_size=128, frame_skip=8, split="holdout", holdout_n=4)
loader = DataLoader(eval_ds, batch_size=16, num_workers=2)

# 跑评估
ev = evaluate(model, effect_tok, loader, "cuda", "cuda", True, cfg)
print("\n=== EVALUATION COMPLETED ===")
print(f"  align (自监督潜对齐 MSE):  {ev['align']:.4f}")
print(f"  agree (多上下文预测一致性): {ev['agree']:.4f}")
print(f"  drift (多步脑内闭环漂移):   {ev['rollout_drift']:.4f}")
print(f"  corr(w, future) (后果权重与潜发散相关度): {ev['corr_w_future']:.3f}")
print(f"  corr(w, pixel) (后果权重与像素差相关度):   {ev['corr_w_pixel']:.3f}")